In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

In [3]:
df = pd.read_csv("../data/diabetes.csv")

In [4]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 769 entries, 0 to 768
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               769 non-null    int64  
 1   Glucose                   769 non-null    int64  
 2   BloodPressure             769 non-null    int64  
 3   SkinThickness             769 non-null    int64  
 4   Insulin                   769 non-null    int64  
 5   BMI                       769 non-null    float64
 6   DiabetesPedigreeFunction  769 non-null    float64
 7   Age                       769 non-null    int64  
 8   Outcome                   769 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.2 KB


In [6]:
df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,769.000000,769.000000,769.000000,769.000000,769.000000,769.000000,769.000000,769.000000,769.000000
mean,3.840052,120.897269,69.115735,20.509753,79.697009,31.998179,0.471590,33.269181,0.349805
std,3.370237,31.951886,19.345296,15.959020,115.203999,7.880557,0.331208,11.778737,0.477219
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.244000,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,29.000000,32.000000,0.371000,29.000000,0.000000
75%,6.000000,140.000000,80.000000,32.000000,127.000000,36.600000,0.626000,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [15]:
df['Outcome'].value_counts()

Outcome
0    500
1    269
Name: count, dtype: int64

In [61]:
df.columns


Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome', 'DPF_cat',
       'DPF_mapped'],
      dtype='object')

In [124]:
X = df[[ 'Glucose', 'BloodPressure',
       'BMI','DPF_mapped', 'Age']]
y= df['Outcome']                                         

In [125]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [126]:
rf = RandomForestClassifier(n_estimators=200,max_depth=8,random_state=42)
xg = XGBClassifier(random_state = 42,
    eval_metric = "logloss",
    use_label_encoder = False)

In [127]:
rf.fit(X_train,y_train)
xg.fit(X_train,y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...)

In [128]:
y_pred_rf = rf.predict(X_test)
y_pred_xg = xg.predict(X_test)

In [129]:
rf_accuracy = accuracy_score(y_test,y_pred_rf)
xg_accuracy = accuracy_score(y_test,y_pred_xg)
print("Rf accuracy", rf_accuracy)
print("xg accuracy", xg_accuracy)

Rf accuracy 0.7857142857142857
xg accuracy 0.7142857142857143


In [130]:
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_
})

feature_importance = feature_importance.sort_values(by='importance', ascending=False)

In [131]:
feature_importance

,feature,importance
0,Glucose,0.368949
2,BMI,0.238173
4,Age,0.214098
1,BloodPressure,0.122242
3,DPF_mapped,0.056539


In [78]:
def categorize_dpf(x):
    if x < 0.3:
        return 0
    elif x < 0.5:
        return 1
    elif x < 0.8:
        return 2
    else:
        return 4

df['DPF_cat'] = df['DiabetesPedigreeFunction'].apply(categorize_dpf)

In [79]:
X.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DPF_mapped,Age
0,6,148,72,35,0,33.6,0.7,50
1,1,85,66,29,0,26.6,0.4,31
2,8,183,64,0,0,23.3,0.7,32
3,1,89,66,23,94,28.1,0.2,21
4,0,137,40,35,168,43.1,1.0,33


In [59]:
dpf_map = {
    0: 0.2,
    1: 0.4,
    2: 0.7,
    4: 1.0
}

df['DPF_mapped'] = df['DPF_cat'].map(dpf_map)

In [80]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome,DPF_cat,DPF_mapped
0,6,148,72,35,0,33.6,0.627,50,1,2,0.7
1,1,85,66,29,0,26.6,0.351,31,0,1,0.4
2,8,183,64,0,0,23.3,0.672,32,1,2,0.7
3,1,89,66,23,94,28.1,0.167,21,0,0,0.2
4,0,137,40,35,168,43.1,2.288,33,1,4,1.0


In [132]:
from sklearn.model_selection import RandomizedSearchCV

In [133]:
param_dist_rf = {
    "n_estimators": np.arange(100, 500, 50),
    "max_depth": [None, 4, 6, 8, 10],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

param_dist_xgb = {
    "n_estimators": np.arange(100, 400, 50),
    "max_depth": [3, 4, 5, 6, 7],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "gamma": [0, 0.1, 0.2, 0.3]
}

In [141]:
rf = RandomForestClassifier(class_weight='balanced',random_state=42)
xgb = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')

In [142]:
rf_random = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist_rf,
    n_iter=20,              # number of combinations to try
    cv=5,                   # 5-fold cross-validation
    verbose=2,
    random_state=42,
    n_jobs=-1
)

rf_random.fit(X_train, y_train)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


RandomizedSearchCV(cv=5,
                   estimator=RandomForestClassifier(class_weight='balanced',
                                                    random_state=42),
                   n_iter=20, n_jobs=-1,
                   param_distributions={'max_depth': [None, 4, 6, 8, 10],
                                        'max_features': ['sqrt', 'log2'],
                                        'min_samples_leaf': [1, 2, 4],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': array([100, 150, 200, 250, 300, 350, 400, 450])},
                   random_state=42, verbose=2)

In [143]:
print("Best Params:", rf_random.best_params_)
print("Best Score:", rf_random.best_score_)

best_rf = rf_random.best_estimator_

Best Params: {'n_estimators': np.int64(400), 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': 6}
Best Score: 0.7642276422764228


In [137]:
xgb_random = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist_xgb,
    n_iter=25,
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

xgb_random.fit(X_train, y_train)

Fitting 5 folds for each of 25 candidates, totalling 125 fits


RandomizedSearchCV(cv=5,
                   estimator=XGBClassifier(base_score=None, booster=None,
                                           callbacks=None,
                                           colsample_bylevel=None,
                                           colsample_bynode=None,
                                           colsample_bytree=None, device=None,
                                           early_stopping_rounds=None,
                                           enable_categorical=False,
                                           eval_metric='logloss',
                                           feature_types=None,
                                           feature_weights=None, gamma=None,
                                           grow_policy=None,
                                           importance_type=None,
                                           interaction_cons...
                                           monotone_constraints=None,
                                           multi_strategy=None,
                                           n_estimators=None, n_jobs=None,
                                           num_parallel_tree=None, ...),
                   n_iter=25, n_jobs=-1,
                   param_distributions={'colsample_bytree': [0.7, 0.8, 0.9,
                                                             1.0],
                                        'gamma': [0, 0.1, 0.2, 0.3],
                                        'learning_rate': [0.01, 0.05, 0.1, 0.2],
                                        'max_depth': [3, 4, 5, 6, 7],
                                        'n_estimators': array([100, 150, 200, 250, 300, 350]),
                                        'subsample': [0.7, 0.8, 0.9, 1.0]},
                   random_state=42, verbose=2)

In [138]:
print("Best Params:", xgb_random.best_params_)
print("Best Score:", xgb_random.best_score_)

best_xgb = xgb_random.best_estimator_

Best Params: {'subsample': 1.0, 'n_estimators': np.int64(300), 'max_depth': 4, 'learning_rate': 0.2, 'gamma': 0.3, 'colsample_bytree': 1.0}
Best Score: 0.7723577235772359


In [144]:
from sklearn.metrics import accuracy_score

rf_pred = best_rf.predict(X_test)
xgb_pred = best_xgb.predict(X_test)

print("RF Accuracy:", accuracy_score(y_test, rf_pred))
print("XGB Accuracy:", accuracy_score(y_test, xgb_pred))

RF Accuracy: 0.7662337662337663
XGB Accuracy: 0.7532467532467533


In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, rf_pred))

              precision    recall  f1-score   support

           0       0.89      0.73      0.80       100
           1       0.62      0.83      0.71        54

    accuracy                           0.77       154
   macro avg       0.76      0.78      0.76       154
weighted avg       0.80      0.77      0.77       154



In [146]:
probs = best_rf.predict_proba(X_test)[:,1]
y_pred_rf_tuned = (probs > 0.45).astype(int)

In [147]:
print(classification_report(y_test, y_pred_rf_tuned))

              precision    recall  f1-score   support

           0       0.88      0.69      0.78       100
           1       0.59      0.83      0.69        54

    accuracy                           0.74       154
   macro avg       0.74      0.76      0.73       154
weighted avg       0.78      0.74      0.75       154

